In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from data_preprocessing import preprocess_data  # noqa: E402

# ── Load sample data ───────────────────────────────────────────────────────────
df = pd.read_csv(ROOT / "data" / "processed" / "fraud_sample.csv")
print(f"Loaded shape: {df.shape}")

# ── Preprocess (drop IDs, one-hot encode, engineer features, fill NaN) ─────────
df = preprocess_data(df)
print(f"After preprocessing: {df.shape}")
print(df.head())

# ── Split features and target ──────────────────────────────────────────────────
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# ── Train / test split (stratified to preserve fraud rate) ────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# ── SMOTE applied ONLY to the training set ────────────────────────────────────
# Reason: applying SMOTE before the split would synthesise minority-class
# examples that overlap with the test set, inflating evaluation metrics.
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print("\nBalanced class distribution:\n", y_train_bal.value_counts())

# Convert back to DataFrame to preserve column names downstream.
X_train_bal = pd.DataFrame(X_train_bal, columns=X_train.columns).astype(float)
X_test = pd.DataFrame(X_test, columns=X_train.columns).astype(float)

# ── Feature scaling (fit on train only) ───────────────────────────────────────
# Note: XGBoost does not require scaling (tree splits are rank-based).
# Scaling is included here for use with linear baseline models (Logistic Regression).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed")

# ── Save processed artifacts ───────────────────────────────────────────────────
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

joblib.dump(X_train_scaled,  processed / "X_train_scaled.pkl")
joblib.dump(X_test_scaled,   processed / "X_test_scaled.pkl")
joblib.dump(X_train_bal,     processed / "X_train_smote.pkl")
joblib.dump(X_test,          processed / "X_test.pkl")
joblib.dump(y_train_bal,     processed / "y_train_smote.pkl")
joblib.dump(y_test,          processed / "y_test.pkl")
joblib.dump(scaler,          processed / "scaler.pkl")
print("All processed artifacts saved to data/processed/")